## Generate Question-Answer Pairs

We will take the chunks of text from the previous step and generate question-answer pairs based on the content of each chunk.

Each chunk can be associated to one or more questions. The questions will be generated by an LLM, but we will need to make sure that the questions are relevant to the content of the chunk.

Run this in terminal to start llama server:
```bash
./llama.cpp/llama-server.exe \
  --model models/Qwen3-4B-Q4_K_M.gguf \
  --ctx-size 8192 \
  --n-gpu-layers 999 \
  --threads 4 \
  --batch-size 2048 \
  --ubatch-size 512 \
  --flash-attn auto \
  --port 36912
```
```bash

In [8]:
import json
import re
import os
import time
import requests
from pathlib import Path
from pydantic import BaseModel
from tqdm import tqdm

# Paths
CHUNKS_PATH   = "../data/chunks/enriched_chunks.json"
OUTPUT_PATH   = "../data/qa_pairs/qa_pairs.json"
PROGRESS_PATH = "../data/qa_pairs/qa_progress.json"

LLAMA_SERVER_URL = "http://localhost:36912"

# Generation Config
MAX_TOKENS   = 2048   # increased — gives model room after thinking tokens
TEMPERATURE  = 0.7    # raised — 0.3 causes Qwen3-4B to get stuck in think loop
TOP_P        = 0.9
MAX_RETRIES  = 3
RETRY_DELAY  = 2      # small delay between retries

In [9]:
#System Prompt
system_prompt = """/no_think
## Task Description

You are an expert at generating relevant question-answer pairs from a vehicle owner's manual (Maruti Suzuki).
Your task is to read a chunk of text and create one or more questions that accurately reflect
the key information contained within that chunk.

These Questions will be used to benchmark embedding models — so questions must be specific enough
that only THIS chunk (or very similar ones) would answer them correctly.

## Rules

- Generate 1 to 4 questions per chunk depending on content richness.
- Each question must be answerable SOLELY from the given chunk — no external knowledge needed.
- Do NOT generate vague or generic questions like "What is this section about?".
- Do NOT generate duplicate or near-duplicate questions.
- Vary question types: factual, procedural, safety-related, specification-based.
- For figure/diagram chunks, generate questions about what the diagram shows or labels.

## Output Format

Return ONLY a valid JSON array. No explanation, no markdown, no preamble.

[
  {
    "question": "A specific question based on the chunk.",
    "question_type": "factual | procedural | safety | specification | figure"
  }
]

## Example

### Chunk:
The seat belt pretensioner system is designed to reduce the slack in the seat belt
during a collision. It activates automatically when the vehicle detects a frontal impact
above a certain threshold. Do not attempt to repair or modify the pretensioner system.

### Output:
[
  {
    "question": "What is the purpose of the seat belt pretensioner system?",
    "question_type": "factual"
  },
  {
    "question": "Under what condition does the seat belt pretensioner activate?",
    "question_type": "factual"
  },
  {
    "question": "What precaution is mentioned regarding the pretensioner system?",
    "question_type": "safety"
  }
]
"""

In [10]:
#Pydantic Models
from pydantic import BaseModel, field_validator
from typing import Literal

class QAPair(BaseModel):
    question: str
    question_type: Literal["factual", "procedural", "safety", "specification", "figure"] = "factual"

    @field_validator("question")
    @classmethod
    def must_not_be_empty(cls, v):
        if not v or not v.strip():
            raise ValueError("Question must not be empty")
        return v.strip()

In [11]:
#Generation Functions
def check_server_health():
    """Check llama.cpp server is running before starting pipeline."""
    try:
        r = requests.get(f"{LLAMA_SERVER_URL}/health", timeout=5)
        if r.status_code == 200:
            print("llama.cpp server is healthy")
            return True
    except Exception:
        pass
    print("llama.cpp server is not reachable. Start it first.")
    return False


def strip_thinking_tokens(text: str) -> str:
    """
    Qwen3 emits <think>...</think> blocks before answering.
    Strip them before JSON parsing.
    """
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def extract_json_from_response(text: str) -> str:
    """
    Robustly extract JSON array from LLM response.
    Handles cases where LLM wraps JSON in markdown code blocks.
    """
    text = strip_thinking_tokens(text)

    # Try extracting from markdown code block
    match = re.search(r"```(?:json)?\s*(\[.*?\])\s*```", text, re.DOTALL)
    if match:
        return match.group(1)

    # Try finding raw JSON array
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        return match.group(0)

    return text


def parse_qa_pairs(raw_response: str) -> list[QAPair]:
    """Parse and validate LLM response into QAPair objects."""
    try:
        cleaned = extract_json_from_response(raw_response)
        data = json.loads(cleaned)
        pairs = []
        for item in data:
            try:
                pairs.append(QAPair(**item))
            except Exception as e:
                print(f"  [SKIP] Invalid QA item: {e} | item: {item}")
        return pairs
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON parse failed: {e}")
        print(f"  Raw response: '{raw_response[:300]}'")   # 🔴 better logging
        return []


def get_user_prompt(chunk: dict) -> str:
    text = chunk.get("contextualized_text") or chunk.get("text", "")
    return f"""Here is a chunk of text from a Maruti Suzuki vehicle owner's manual:

{text}

Generate relevant questions based on the content of this chunk. /no_think
"""
#                                                             ^^^^^^^^^^
# 🔴 /no_think disables Qwen3 extended thinking mode
# prevents thinking tokens from eating the entire token budget


def generate_qa_pairs(chunk: dict, retries: int = MAX_RETRIES) -> list[QAPair]:
    """Call llama.cpp server with retry logic."""
    url     = f"{LLAMA_SERVER_URL}/v1/chat/completions"
    headers = {"Content-Type": "application/json"}

    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": get_user_prompt(chunk)},
        ],
        "max_tokens"  : MAX_TOKENS,
        "temperature" : TEMPERATURE,
        "top_p"       : TOP_P,
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                url=url,
                headers=headers,
                json=payload,
                timeout=120    # 🔴 increased timeout for larger token budget
            )
            response.raise_for_status()

            raw = response.json()

            # 🔴 check finish reason — warn if truncated
            finish_reason = raw["choices"][0].get("finish_reason", "")
            if finish_reason == "length":
                print(f"  [WARN] Response truncated at max_tokens for chunk {chunk.get('chunk_id')}")

            content = raw["choices"][0]["message"]["content"].strip()

            if not content:
                print(f"  [WARN] Empty response for chunk {chunk.get('chunk_id')} — skipping")
                return []

            return parse_qa_pairs(content)

        except requests.exceptions.Timeout:
            print(f"  [TIMEOUT] Attempt {attempt}/{retries}")
        except requests.exceptions.RequestException as e:
            print(f"  [REQUEST ERROR] Attempt {attempt}/{retries}: {e}")
        except Exception as e:
            print(f"  [ERROR] Attempt {attempt}/{retries}: {e}")

        if attempt < retries:
            time.sleep(RETRY_DELAY)

    return []

In [12]:
# Load chunks
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"Total chunks: {len(all_chunks)}")

# Load progress if resuming
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

if Path(PROGRESS_PATH).exists():
    with open(PROGRESS_PATH, "r") as f:
        progress = json.load(f)
    done_chunk_ids = set(progress["done_chunk_ids"])
    qa_pairs       = progress["qa_pairs"]
    print(f"Resuming — {len(done_chunk_ids)} chunks already processed, {len(qa_pairs)} QA pairs so far")
else:
    done_chunk_ids = set()
    qa_pairs       = []
    print("Starting fresh")

Total chunks: 3055
Starting fresh


In [ ]:
# Run Pipeline
assert check_server_health(), "Fix server before running pipeline"

SAVE_EVERY = 20

for i, chunk in enumerate(tqdm(all_chunks, desc="Generating QA pairs")):

    chunk_id = chunk["chunk_id"]

    if chunk_id in done_chunk_ids:
        continue

    pairs = generate_qa_pairs(chunk)

    for q_idx, pair in enumerate(pairs):
        qa_pairs.append({
            "qa_id"         : f"{chunk_id}_q{q_idx}",   # unique id
            "chunk_id"      : chunk_id,
            "page_no"       : chunk.get("page_no"),
            "question"      : pair.question,
            "question_type" : pair.question_type,
        # useful metadata for debugging / eval
            "chunk_text"          : chunk.get("text"),
            "contextualized_text" : chunk.get("contextualized_text"),
            "heading"       : chunk.get("heading"),
            "is_figure"     : chunk.get("is_figure_chunk", False),
    })

    done_chunk_ids.add(chunk_id)

    if (i + 1) % SAVE_EVERY == 0:
        with open(PROGRESS_PATH, "w") as f:
            json.dump({"done_chunk_ids": list(done_chunk_ids), "qa_pairs": qa_pairs}, f)
        print(f"  Progress saved at chunk {i+1}")

# ── Save final progress BEFORE writing output ────────────────────────────────
# Prevents losing the last batch if output save crashes
with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "done_chunk_ids": sorted(list(done_chunk_ids)),  # sorted = readable
            "qa_pairs": qa_pairs
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"\nTotal QA pairs: {len(qa_pairs)}")

llama.cpp server is healthy


Generating QA pairs:   4%|▍         | 123/3055 [00:16<06:23,  7.64it/s]


KeyboardInterrupt: 

In [ ]:
# Save final QA pairs
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

# Clean up progress file only after successful output save
if Path(PROGRESS_PATH).exists():
    Path(PROGRESS_PATH).unlink()

print(f"Saved {len(qa_pairs)} QA pairs → {OUTPUT_PATH}")

from collections import Counter
types = Counter(p["question_type"] for p in qa_pairs)
print("\nQuestion type breakdown:")
for qtype, count in types.most_common():
    print(f"  {qtype:<15} : {count}")